In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv("crop_yield_cleaned_dataset.csv")
df

,Record_ID,Year,Season,State,Latitude,Longitude,Elevation_m,Crop_Type,Irrigation_Method,Fertilizer_Type,...,K_kgha,Sulfur_kgha,Zinc_ppm,Iron_ppm,NDVI,Pest_Incidence,Disease_Incidence,Crop_Yield_kg_ha,Yield_Category,Data_Source
0,AGR2006RIC00000,2006,Kharif,Tamil Nadu,10.8591,78.7410,23.0,Rice,Drip,Mop,...,107.12,14.29,1.08,10.12,0.464,Moderate,Moderate,3156.36,High,FAO-FAOSTAT/ICRISAT-DLD/NASA-POWER/ISRIC-SoilG...
1,AGR2006RIC00001,2006,Kharif,West Bengal,23.8281,87.6540,-16.3,Rice,Furrow,Dap,...,56.92,19.94,1.77,11.95,0.391,Low,Low,2707.44,Medium,FAO-FAOSTAT/ICRISAT-DLD/NASA-POWER/ISRIC-SoilG...
2,AGR2007RIC00002,2007,Kharif,Andhra Pradesh,17.5375,79.6536,140.1,Rice,Drip,Npk Complex,...,61.89,30.31,2.93,5.80,0.434,Low,NaN,2742.57,Medium,FAO-FAOSTAT/ICRISAT-DLD/NASA-POWER/ISRIC-SoilG...
3,AGR2013RIC00003,2013,Kharif,Punjab,30.4871,76.7609,231.9,Rice,Furrow,Dap,...,61.92,42.87,2.04,14.12,0.553,Low,NaN,3422.19,High,FAO-FAOSTAT/ICRISAT-DLD/NASA-POWER/ISRIC-SoilG...
4,AGR2019RIC00004,2019,Kharif,Odisha,21.3119,85.0583,13.3,Rice,Drip,Organic,...,124.76,10.57,3.40,9.91,0.609,NaN,High,3007.44,High,FAO-FAOSTAT/ICRISAT-DLD/NASA-POWER/ISRIC-SoilG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219995,AGR2003MUS21995,2003,Rabi,Rajasthan,27.1235,75.2363,391.1,Mustard,Rainfed,Mixed,...,34.52,26.81,3.75,10.49,0.657,Low,Moderate,427.60,Low,FAO-FAOSTAT/ICRISAT-DLD/NASA-POWER/ISRIC-SoilG...
219996,AGR2004MUS21996,2004,Rabi,Haryana,27.6502,76.2633,216.2,Mustard,Furrow,Npk Complex,...,68.02,17.26,1.55,14.56,0.691,Moderate,Moderate,424.71,Low,FAO-FAOSTAT/ICRISAT-DLD/NASA-POWER/ISRIC-SoilG...
219997,AGR2006MUS21997,2006,Rabi,Uttar Pradesh,26.1012,80.4153,209.6,Mustard,Drip,Npk Complex,...,7.78,22.95,1.46,10.35,0.549,Moderate,High,481.83,Medium,FAO-FAOSTAT/ICRISAT-DLD/NASA-POWER/ISRIC-SoilG...
219998,AGR2018MUS21998,2018,Rabi,Madhya Pradesh,24.0196,77.8386,459.8,Mustard,Rainfed,Npk Complex,...,22.54,17.03,2.84,10.16,0.904,High,High,629.55,High,FAO-FAOSTAT/ICRISAT-DLD/NASA-POWER/ISRIC-SoilG...


In [4]:
# =============================================================================
#  PHASE 3 — DATA PREPROCESSING  (scikit-learn)
#  Project : Crop Yield Prediction Using Environmental and Nutrient Factors
#  Course  : Data Warehousing and Data Mining (DWDM)
#  Tools   : scikit-learn — StandardScaler, MinMaxScaler, RobustScaler,
#             LabelEncoder, OrdinalEncoder, OneHotEncoder,
#             train_test_split, ColumnTransformer, Pipeline
# =============================================================================
 
import pandas as pd
import numpy as np

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    LabelEncoder,
    OrdinalEncoder,
    OneHotEncoder
)
from sklearn.model_selection import train_test_split
from sklearn.compose   import ColumnTransformer
from sklearn.pipeline  import Pipeline
 
df["Pest_Incidence"]    = df["Pest_Incidence"].fillna("None")
df["Disease_Incidence"] = df["Disease_Incidence"].fillna("None")
 
print(f"   Nulls   : {df.isnull().sum().sum()}")

   Shape   : 220,000 rows × 36 columns
   Nulls   : 0


In [5]:
# STEP 1 — DROP COLUMNS NOT USEFUL FOR MODELLING

drop_cols = ["Record_ID", "Data_Source"]
df.drop(columns=drop_cols, inplace=True)
 
# print(f"   Dropped : {drop_cols}")
# print(f"   Shape after drop : {df.shape}")

   Dropped : ['Record_ID', 'Data_Source']
   Shape after drop : (220000, 34)


In [6]:
# STEP 2 — SEPARATE FEATURES AND TARGETS

 # Two targets:
#   Crop_Yield_kg_ha :  continuous  -> use for REGRESSION  task
#   Yield_Category   :  categorical -> use for CLASSIFICATION task
 
TARGET_REG   = "Crop_Yield_kg_ha"   
TARGET_CLASS = "Yield_Category"     
 
X = df.drop(columns=[TARGET_REG, TARGET_CLASS])
y_reg   = df[TARGET_REG]             
y_class = df[TARGET_CLASS]           

   Feature matrix X       : (220000, 32)
   Regression target      : 'Crop_Yield_kg_ha'  → float64
   Classification target  : 'Yield_Category' → {'High': 74799, 'Low': 72601, 'Medium': 72600}


In [8]:
# STEP 3 — IDENTIFY COLUMN TYPES

numerical_cols = [
    "Year",
    "Latitude", "Longitude", "Elevation_m",
    "Avg_Temp_C", "Max_Temp_C", "Min_Temp_C",
    "Rainfall_mm", "Humidity_Pct", "Solar_Radiation_MJm2",
    "Wind_Speed_kmh", "ET0_mm_day", "Water_Stress_Index",
    "Soil_pH", "Soil_Moisture_Pct", "Organic_Carbon_Pct",
    "Bulk_Density_gcm3",
    "N_kgha", "P_kgha", "K_kgha",
    "Sulfur_kgha", "Zinc_ppm", "Iron_ppm",
    "NDVI"
]
 
# Ordinal features 
# Pest_Incidence and Disease_Incidence:  None < Low < Moderate < High
ordinal_cols    = ["Pest_Incidence", "Disease_Incidence"]
ordinal_order   = [["None", "Low", "Moderate", "High"]]   
 
# Nominal features 
nominal_cols = [
    "Season",
    "State",
    "Crop_Type",
    "Irrigation_Method",
    "Fertilizer_Type",
    "Soil_Type"
]
 
# print(f"\n Numerical columns  ({len(numerical_cols)}) :")
# print(f"\n   Ordinal columns    ({len(ordinal_cols)}) :")
# print(f"   {ordinal_cols}  ->  order: None < Low < Moderate < High")
# print(f"\n   Nominal columns    ({len(nominal_cols)}) :")
# print(f"   {nominal_cols}  ->  One-Hot Encoded")


   Numerical columns  (24) :

   Ordinal columns    (2) :
   ['Pest_Incidence', 'Disease_Incidence']  →  order: None < Low < Moderate < High

   Nominal columns    (6) :
   ['Season', 'State', 'Crop_Type', 'Irrigation_Method', 'Fertilizer_Type', 'Soil_Type']  →  One-Hot Encoded


In [9]:
# STEP 4 — ENCODE TARGET VARIABLE (Classification)

label_map   = {"Low": 0, "Medium": 1, "High": 2}
y_class_enc = y_class.map(label_map)
 
# print(f"   Before : {y_class.value_counts().to_dict()}")
# print(f"   After  : {y_class_enc.value_counts().sort_index().to_dict()}")
 
le = LabelEncoder()
le.fit(["Low", "Medium", "High"])
# print(f"   LabelEncoder classes : {le.classes_}")

   Encoding scheme : Low=0  |  Medium=1  |  High=2
   Before : {'High': 74799, 'Low': 72601, 'Medium': 72600}
   After  : {0: 72601, 1: 72600, 2: 74799}
   LabelEncoder classes : ['High' 'Low' 'Medium']


In [10]:
# STEP 5 — TRAIN-TEST SPLIT  (before any scaling)

# stratify=y_class → ensures Low/Medium/High ratio same in train and test
 
X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = \
    train_test_split(
        X, y_reg, y_class_enc,
        test_size    = 0.20,
        random_state = 42,
        stratify     = y_class_enc
    )
 
# print(f"\n   Total samples  : {len(X):,}")
# print(f"   Train set      : {len(X_train):,}  (80%)")
# print(f"   Test  set      : {len(X_test):,}   (20%)")
# print(f"\n   Stratification check (Yield_Category distribution) :")
# print(f"   Train → {pd.Series(y_cls_train).value_counts().sort_index().to_dict()}")
# print(f"   Test  → {pd.Series(y_cls_test).value_counts().sort_index().to_dict()}")


   Total samples  : 220,000
   Train set      : 176,000  (80%)
   Test  set      : 44,000   (20%)

   Stratification check (Yield_Category distribution) :
   Train → {0: 58081, 1: 58080, 2: 59839}
   Test  → {0: 14520, 1: 14520, 2: 14960}


In [11]:
 # STEP 6 — SCALING NUMERICAL FEATURES
 

robust_scaler = RobustScaler()
 
X_train_num = X_train[numerical_cols].copy()
X_test_num  = X_test[numerical_cols].copy()
 
X_train_num_scaled = robust_scaler.fit_transform(X_train_num)   
X_test_num_scaled  = robust_scaler.transform(X_test_num)         #
 

X_train_num_scaled = pd.DataFrame(X_train_num_scaled,columns=numerical_cols,index=X_train.index)
X_test_num_scaled  = pd.DataFrame(X_test_num_scaled,columns=numerical_cols,index=X_test.index)
 

# sample_cols = ["Avg_Temp_C", "Rainfall_mm", "N_kgha", "Crop_Yield_kg_ha"]
# print(f"\n   {'Column':<25} {'Before Mean':>12} {'Before Std':>11} {'After Mean':>11} {'After Std':>10}")
# for col in ["Avg_Temp_C", "Rainfall_mm", "N_kgha"]:
#     bm = X_train[col].mean()
#     bs = X_train[col].std()
#     am = X_train_num_scaled[col].mean()
#     as_ = X_train_num_scaled[col].std()
#     print(f"   {col:<25} {bm:>12.2f} {bs:>11.2f} {am:>11.4f} {as_:>10.4f}")

   RobustScaler fitted on train (176,000 rows)
   Numerical columns scaled : 24

   Before vs After scaling (train set sample) :

   Column                     Before Mean  Before Std  After Mean  After Std
   ───────────────────────── ──────────── ─────────── ─────────── ──────────
   Avg_Temp_C                       23.64        9.24      0.0036     0.6939
   Rainfall_mm                     131.39       83.65      0.1756     0.8197
   N_kgha                          102.99       63.34     -0.0477     0.6343


In [13]:
# STEP 7 — ENCODE ORDINAL FEATURES

ordinal_encoder = OrdinalEncoder(
    categories=[["None", "Low", "Moderate", "High"],
                ["None", "Low", "Moderate", "High"]],
    handle_unknown="use_encoded_value",unknown_value=-1)
 
X_train_ord = X_train[ordinal_cols].copy()
X_test_ord  = X_test[ordinal_cols].copy()
 
X_train_ord_enc = ordinal_encoder.fit_transform(X_train_ord)
X_test_ord_enc  = ordinal_encoder.transform(X_test_ord)
 
X_train_ord_enc = pd.DataFrame(X_train_ord_enc,columns=ordinal_cols,index=X_train.index)
X_test_ord_enc  = pd.DataFrame(X_test_ord_enc,columns=ordinal_cols,index=X_test.index)
 
# for col, cats in zip(ordinal_cols, ordinal_encoder.categories_):
#     print(f"   {col:<22} : {list(enumerate(cats))}")

   Mapping applied:
   Pest_Incidence         : [(0, 'None'), (1, 'Low'), (2, 'Moderate'), (3, 'High')]
   Disease_Incidence      : [(0, 'None'), (1, 'Low'), (2, 'Moderate'), (3, 'High')]

   Sample encoding (first 5 rows of train) :
Pest_Incidence Disease_Incidence  Pest_Incidence_enc  Disease_Incidence_enc
          None          Moderate                 0.0                    2.0
      Moderate              None                 2.0                    0.0
           Low              None                 1.0                    0.0
          High              High                 3.0                    3.0
          None              High                 0.0                    3.0


In [14]:
# STEP 8 — ENCODE NOMINAL FEATURES

ohe = OneHotEncoder(
    drop="first",
    sparse_output=False,
    handle_unknown="ignore"
)
 
X_train_nom = X_train[nominal_cols].copy()
X_test_nom  = X_test[nominal_cols].copy()
 
X_train_nom_enc = ohe.fit_transform(X_train_nom)
X_test_nom_enc  = ohe.transform(X_test_nom)
 
ohe_feature_names = ohe.get_feature_names_out(nominal_cols)
 
X_train_nom_enc = pd.DataFrame(X_train_nom_enc,columns=ohe_feature_names,index=X_train.index)
X_test_nom_enc  = pd.DataFrame(X_test_nom_enc,columns=ohe_feature_names,index=X_test.index)
 
print(f"   Original nominal columns  : {len(nominal_cols)}")
print(f"   One-Hot encoded columns   : {len(ohe_feature_names)}")
# print(f"\n   Columns generated per feature :")
# for col in nominal_cols:
#     cols_for = [c for c in ohe_feature_names if c.startswith(col + "_")]
#     print(f"   {col:<22} → {len(cols_for)} columns  {cols_for[:3]}{'...' if len(cols_for)>3 else ''}")

   Original nominal columns  : 6
   One-Hot encoded columns   : 41

   Columns generated per feature :
   Season                 → 2 columns  ['Season_Kharif', 'Season_Rabi']
   State                  → 14 columns  ['State_Assam', 'State_Bihar', 'State_Gujarat']...
   Crop_Type              → 9 columns  ['Crop_Type_Cotton', 'Crop_Type_Groundnut', 'Crop_Type_Maize']...
   Irrigation_Method      → 4 columns  ['Irrigation_Method_Flood', 'Irrigation_Method_Furrow', 'Irrigation_Method_Rainfed']...
   Fertilizer_Type        → 5 columns  ['Fertilizer_Type_Mixed', 'Fertilizer_Type_Mop', 'Fertilizer_Type_Npk Complex']...
   Soil_Type              → 7 columns  ['Soil_Type_Clay', 'Soil_Type_Clay Loam', 'Soil_Type_Loamy']...


In [15]:
# STEP 9 — COMBINE ALL PREPROCESSED FEATURES

X_train_final = pd.concat([X_train_num_scaled,X_train_ord_enc,X_train_nom_enc], axis=1)
 
X_test_final  = pd.concat([X_test_num_scaled,X_test_ord_enc,X_test_nom_enc], axis=1)


   Numerical (scaled)     : 24 columns
   Ordinal (encoded)      : 2 columns
   Nominal (one-hot)      : 41 columns
   ───────────────────────────────────
   Total features         : 67 columns

   X_train_final shape    : (176000, 67)
   X_test_final  shape    : (44000, 67)


In [16]:
# STEP 10 — SCIKIT-LEARN PIPELINE  (production-ready version)

preprocessor = ColumnTransformer(transformers=[
    ("num",     RobustScaler(),numerical_cols),
    ("ordinal", OrdinalEncoder(
                    categories=[["None","Low","Moderate","High"],
                                ["None","Low","Moderate","High"]],
                    handle_unknown="use_encoded_value",
                    unknown_value=-1),
                ordinal_cols),
    ("nominal", OneHotEncoder(drop="first", sparse_output=False,
                              handle_unknown="ignore"),
                nominal_cols),
], remainder="drop")
 
X_train_pipe = preprocessor.fit_transform(X_train)
X_test_pipe  = preprocessor.transform(X_test)
 
print(f"   Pipeline fitted on train set")
print(f"   X_train_pipe shape : {X_train_pipe.shape}")
print(f"   X_test_pipe  shape : {X_test_pipe.shape}")

   Pipeline fitted on train set
   X_train_pipe shape : (176000, 67)
   X_test_pipe  shape : (44000, 67)


   Output arrays ready for model training :
   X_train_pipe  → shape (176000, 67)  (use for model.fit)
   X_test_pipe   → shape (44000, 67)   (use for model.predict)
   y_reg_train   → shape (176000,)      (regression target train)
   y_reg_test    → shape (44000,)       (regression target test)
   y_cls_train   → shape (176000,)      (classification target train)
   y_cls_test    → shape (44000,)       (classification target test)


In [19]:
# SAVE PREPROCESSED DATA


num_feat_names = numerical_cols
ord_feat_names = ordinal_cols
nom_feat_names = list(preprocessor.named_transformers_["nominal"]
                      .get_feature_names_out(nominal_cols))
all_feat_names = num_feat_names + ord_feat_names + nom_feat_names
 
train_out = pd.DataFrame(X_train_pipe, columns=all_feat_names)
train_out["Crop_Yield_kg_ha"] = y_reg_train.values
train_out["Yield_Category"]   = y_cls_train.values
 
test_out  = pd.DataFrame(X_test_pipe,  columns=all_feat_names)
test_out["Crop_Yield_kg_ha"]  = y_reg_test.values
test_out["Yield_Category"]    = y_cls_test.values
 
train_out.to_csv("crop_yield_train_preprocessed.csv", index=False)
test_out.to_csv("crop_yield_test_preprocessed.csv",   index=False)
 
# print(f"\n Saved → crop_yield_train_preprocessed.csv  ({len(train_out):,} rows)")
# print(f" Saved → crop_yield_test_preprocessed.csv   ({len(test_out):,} rows)")


✅ Saved → crop_yield_train_preprocessed.csv  (176,000 rows)
✅ Saved → crop_yield_test_preprocessed.csv   (44,000 rows)
